# Football Managerial Decision Analytics — Chapter 05
## Statistical inference, uncertainty, clustering and models

> **Data.** StatsBomb Open Data (https://github.com/hudl/open-data), provided by StatsBomb / Hudl, used for educational, non-commercial research. Raw data are downloaded from the official repository at run time and are not redistributed here. Every figure carries its chart ID from `docs/VISUALIZATION_CATALOGUE.md`.

This notebook is one chapter of the research record. All logic lives in the shared `src/` package; the cells call it. The same charts are available interactively in the Streamlit dashboard (`streamlit run app.py`).

In [ ]:
# Setup — locate (or fetch) the project code, import the shared package, load the workspace
import os, subprocess, sys, time
from pathlib import Path

REPO_URL = "https://github.com/blazingarrows1525/football-managerial-decision-analytics"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "workspace.py").exists()), None)
if ROOT is None:  # opened on its own (e.g. Google Colab): fetch the project code once, next to this notebook
    ROOT = Path.cwd() / "football-managerial-decision-analytics"
    if not (ROOT / "src" / "workspace.py").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("FMA_PROJECT_ROOT", str(ROOT))  # data cache and outputs live inside the project folder

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src.reporting.notebook_tools import (catalogue, explorer, implication, interpret, limitation, show, start,
                                          timed, timings)

pd.set_option("display.max_columns", 40, "display.width", 200, "display.float_format", "{:.3f}".format)
RUN_ADVANCED_GALLERY = True   # heavier, optional sections; the core story does not depend on them
SEED = 42                             # every resampling and model step takes this seed explicitly

ws = start(seed=SEED)                 # first run downloads and parses; later runs read the processed cache
B, T, FOCUS = ws.bundle, ws.tables, ws.focus
tm = ws.team_matches()                # the focus team's matches in date order
print(f"Advanced sections: {'on' if RUN_ADVANCED_GALLERY else 'off'} · Python {sys.version.split()[0]} · pandas {pd.__version__}")

## G · Statistical inference and uncertainty

**Research question.** Which relationships are in the data, how large are they, and how much could be chance?

**Method.** Rank-based statistics by default (the metrics are skewed), effect sizes and bootstrap intervals
next to every p-value, Benjamini–Hochberg control across a family of tests, and resampling at the level of the
independent unit — matches for team questions, players for player questions. Events are never treated as
independent observations.

In [ ]:
from src.analytics import statistical_tests as stt
corr = ws.correlations()
display(corr.round(3))
show("J-202", ws)
show("J-213", ws)
top = corr.iloc[0]; poss = corr.set_index("variable").loc["possession_proxy_pct"]
n_sig = int((corr.spearman_q_BH < 0.05).sum())
interpret(f"Of {len(corr)} match variables, {n_sig} remain associated with goal difference after false-discovery control "
          f"(n = {int(top.n)} matches). The strongest is {top.variable} (ρ = {top.spearman_rho:.2f}, 95% CI {top.rho_ci_low:.2f} to "
          f"{top.rho_ci_high:.2f}, q = {top.spearman_q_BH:.3f}); possession proxy shows ρ = {poss.spearman_rho:.2f} (q = {poss.spearman_q_BH:.3f}).")

In [ ]:
show("J-214", ws)
show("L-276", ws)
mdr = float(stt.min_detectable_correlation(len(tm)))
interpret(f"With n = {len(tm)} a two-sided test at α = 0.05 has 80% power only for |r| ≥ {mdr:.2f}. Weaker real "
          f"associations will usually be missed, so a non-significant result here is not evidence of no relationship. "
          f"H1 (chance quality matters more than possession) is supported in direction and "
          + ("in significance." if n_sig else "undecided in significance at this sample size."))
limitation("One team-season: opponents and venues are not randomised, and nothing here is a causal estimate.")

In [ ]:
tests = ws.group_tests()
display(pd.DataFrame(tests)[["comparison", "n_a", "n_b", "mean_a", "mean_b", "mean_diff", "diff_ci_low", "diff_ci_high",
                             "p_value", "effect_size", "magnitude"]].round(3))
show("J-211", ws)
show("J-209", ws, team_metric="xg_diff")
v = tests[0]
interpret(f"Home minus away xG difference is {v['mean_diff']:+.2f} (95% CI {v['diff_ci_low']:+.2f} to {v['diff_ci_high']:+.2f}), "
          f"Cliff's δ = {v['effect_size']:.2f} ({v['magnitude']}), p = {v['p_value']:.3f} — "
          + ("statistically significant." if v['p_value'] < 0.05 else "suggestive but not established; the interval includes zero."))
implication("Not strong enough to justify a different game model by venue on this evidence.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    show("J-212", ws, metric="progressive_passes_p90", group="role_family")
    show("J-207", ws, team_metric="xg_diff")
    show("J-219", ws, team_metric="xg_diff")
    show("J-215", ws, metric="xg_p90", metric_y="key_passes_p90")
    show("J-216", ws)
    show("J-217", ws, team_metric="xg_diff")

## H · Unsupervised structure: PCA and clustering

**Question.** Do player profiles form distinct types, or a continuum?

**Method.** Fourteen per-90 metrics, standardised, for outfield players above the minutes threshold. Four
algorithms (K-Means, Ward, Gaussian mixture, DBSCAN) and four selection criteria, plus a bootstrap stability
check — so the conclusion does not rest on one method.

In [ ]:
pca = ws.pca(); cl = ws.clusters()
show("K-237", ws)
show("K-240", ws)
show("L-277", ws)
show("K-242", ws)
stab = ws.cluster_stability()
interpret(f"PC1 and PC2 explain {pca['explained'][0]:.0%} and {pca['explained'][1]:.0%} of the variance across {pca['n_players']} "
          f"players; {pca['components_for_80pct']} components are needed for 80%. Silhouette is best at k = {cl['k_best']} "
          f"({cl['silhouette_best']:.2f}) while the mixture-model BIC prefers k = {cl['k_bic']}; DBSCAN finds "
          f"{cl['dbscan_clusters']} dense group(s) and {cl['dbscan_noise']} outlying players. Bootstrap stability of the "
          f"K-Means partition: mean ARI {stab.ari.mean():.2f}.")
implication("Use role-specific metric profiles for squad planning; cluster labels are a vocabulary, not fixed "
            "player types.")
limitation("Opponent players are observed only in their two matches against the focus team.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    show("K-238", ws)
    show("K-239", ws)
    show("K-241", ws)
    show("L-278", ws)
    show("K-243", ws)

## Supervised modelling — only where the data support it

**Question.** Can the starting decision be predicted from information available before kick-off, and does a
model beat a simple rule?

**Target.** One row per match-day squad member of the focus team; y = 1 if he started. **Baseline:** he started
the previous match. **Split:** by match in date order (expanding-window cross-validation and a final
chronological hold-out) so no future information reaches a prediction.

**Not modelled:** match results (34 matches, no losses), goals, injuries, fitness, market value — the data do
not support them.

In [ ]:
from src.analytics import models
with timed("model comparison"):
    res = ws.models()
display(pd.DataFrame([res["gate"]]))
display(pd.DataFrame(models.UNSUPPORTED_TARGETS, columns=["target not modelled", "why"]))
if res["fitted"]:
    display(res["leakage_audit"])
    cs = res["cv_summary"]; display(cs.round(3))
    show("K-221", ws); show("K-236", ws)
    f1 = cs[("f1", "mean")]; auc = cs[("roc_auc", "mean")].dropna()
    interpret(f"Time-ordered cross-validation on {res['gate']['rows']} rows from {res['gate']['matches']} matches: F1 "
              + ", ".join(f"{m} {v:.3f}" for m, v in f1.items())
              + f". ROC-AUC ranges from {auc.min():.3f} to {auc.max():.3f}. The best model improves on the baseline rule by "
              f"{f1.drop(models.BASELINE_NAME).max() - f1[models.BASELINE_NAME]:.3f} F1.")
else:
    interpret(f"No model is fitted: {res['reason']}")

In [ ]:
if res["fitted"]:
    s = res["split"]; hold = res["holdout"].set_index("model")
    print(f"Hold-out: train {s['train_matches']} matches to {s['train_until']:%d %b %Y}, test {s['test_matches']} matches from {s['test_from']:%d %b %Y}")
    display(res["holdout"].round(3))
    show("K-223", ws); show("K-225", ws); show("K-228", ws)
    perm = ws.permutation()
    interpret(f"On the chronological hold-out {res['best']} ranks starters best (ROC-AUC {hold.loc[res['best'], 'roc_auc']:.3f}, "
              f"Brier {hold.loc[res['best'], 'brier']:.3f}). Shuffling {perm.feature.iloc[-1]} costs {perm.importance_mean.iloc[-1]:.3f} AUC "
              f"and {perm.feature.iloc[-2]} {perm.importance_mean.iloc[-2]:.3f}: selection follows past selection.")
    implication("A rule plus season minutes share explains most selections. Analytics adds value in the rotation "
                "margin — where the missing availability data matter most.")
    limitation("Describes this team-season only; importance is conditional on correlated features and is not a "
               "causal lever.")

In [ ]:
# Optional, heavier section — set RUN_ADVANCED_GALLERY = True in the setup cell to run it
if RUN_ADVANCED_GALLERY:
    if res["fitted"]:
        for cid in ("K-224", "K-222", "K-226", "K-227", "K-232", "K-233", "K-234", "K-231", "K-229", "K-235"):
            show(cid, ws)